In [0]:
df = spark.read.parquet("abfss://bronze@ttadlsjcrh.dfs.core.windows.net/npi_extract")
df.createOrReplaceTempView('npi_extract')

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5973047298564730>, line 2
      1 df=spark.read.parquet("abfss://bronze@ttadlsjcrh.dfs.core.windows.net/npi_extract")
----> 2 df.createOrReplaceTempView('npi_extract')

File /databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/dataframe.py:2026, in DataFrame.createOrReplaceTempView(self, name)
   2022 def createOrReplaceTempView(self, name: str) -> None:
   2023     command = plan.CreateView(
   2024         child=self._plan, name=name, is_global=False, replace=True
   2025     ).command(session=self._session.client)
-> 2026     self._session.client.execute_command(command, self._plan.observations)

File /databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/client/core.py:1208, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1206     req.user_contex

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.npi_extract (
  npi_id STRING,
  first_name STRING,
  last_name STRING,
  position STRING,
  organisation_name STRING,
  last_updated STRING,
  inserted_date DATE,
  updated_date DATE,
  is_current_flag BOOLEAN
)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:717)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
%sql
MERGE INTO
  silver.npi_extract AS target
USING
  npi_extract AS source
ON target.npi_id = source.npi_id and target.is_current_flag = true
WHEN MATCHED AND
  target.first_name != source.first_name OR
  target.last_name != source.last_name OR
  target.position != source.position OR
  target.organisation_name != source.organisation_name OR
  target.last_updated != source.last_updated
  THEN UPDATE SET
  target.updated_date = current_date,
  target.is_current_flag = False


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
MERGE INTO
  silver.npi_extract AS target
USING
  npi_extract AS source
ON target.npi_id = source.npi_id and target.is_current_flag = true
WHEN NOT MATCHED THEN INSERT (
   npi_id,
  first_name ,
  last_name ,
  position ,
  organisation_name ,
  last_updated ,
  inserted_date ,
  updated_date ,
  is_current_flag 
)
  VALUES (
    source.npi_id,
  source.first_name ,
  source.last_name ,
  source.position ,
  source.organisation_name ,
  source.last_updated ,
  current_date,
  current_date, 
  true
  )

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
20,0,0,20


In [0]:
%python
df_sil_npi_extract = spark.read.format("delta").table("silver.npi_extract")

# Define your ADLS Gen2 path
adls_path = "abfss://silver@ttadlsjcrh.dfs.core.windows.net/"

# Write the DataFrame as a Delta table
df_sil_npi_extract.write.format("delta").mode("overwrite").save(adls_path + "/npi_extract")

In [0]:
%sql
select * from silver.npi_extract